# BioEmu Conformational Ensemble Sampling

![BioEmu Conformational Ensemble Sampling](https://proto-bio.github.io/proto-assets/images/tool/bioemu/hero.png)

This notebook demonstrates conformational ensemble sampling using BioEmu. We generate a set of backbone conformations for a short protein sequence, inspect the resulting ensemble, and demonstrate batch sampling across multiple complexes in a single call.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("bioemu")
display_overview("bioemu")
display_docs_section("bioemu", "Background")

## Available tools

In [ ]:
display_available_tools("bioemu")

### `run_bioemu`

BioEmu generates protein conformational ensembles by sampling from a learned distribution over backbone conformations. It accepts one or more single-chain protein complexes and returns an ensemble of backbone structures for each, representing the thermodynamic diversity of conformations that the protein is likely to adopt. BioEmu always runs MSA search via ColabFold before inference, unless pre-computed MSAs are already attached to the input. Multiple complexes can be submitted in a single call; the model processes them together and returns one ensemble per input.

In [ ]:
from proto_tools import (
    BioEmuConfig,
    BioEmuInput,
    Complex,
    run_bioemu,
)

In [ ]:
# Display input docs
display_api_reference("bioemu", "input", "run_bioemu")

# Chignolin (CLN025) is a 10-residue fast-folding beta-hairpin, a classic benchmark
# for conformational ensemble sampling.
complex_ = Complex(
    chains=[{"sequence": "YYDPETGTWY", "entity_type": "protein"}]
)

inputs = BioEmuInput(complexes=[complex_])

In [ ]:
# Display config docs
display_api_reference("bioemu", "config", "run_bioemu")

# Sample 50 conformations using the v1.1 model with quality filtering enabled
config = BioEmuConfig(
    num_samples=50,
    model_name="bioemu-v1.1",
    filter_samples=True,
    device="cuda",  # Change to "cpu" if no GPU available
)

In [ ]:
# Run the sampling
single_result = run_bioemu(inputs, config)

In [ ]:
# Display output docs and inspect results
display_api_reference("bioemu", "output", "run_bioemu")

print(f"Ensembles: {len(single_result.ensembles)}")
print(f"Conformations: {len(single_result.ensembles[0].structures)}")

### Batch sampling across multiple complexes

Multiple complexes can be submitted in a single call, which is more efficient than making separate calls because MSA search and GPU allocation happen only once. Each complex produces its own ensemble in the output.

In [ ]:
from proto_tools import StructureMetricsConfig, StructureMetricsInput, run_structure_metrics

# Two real fast-folding mini-proteins with well-characterized conformational dynamics.
chignolin = Complex(chains=[{"sequence": "YYDPETGTWY", "entity_type": "protein"}])  # CLN025 beta-hairpin
trp_cage = Complex(chains=[{"sequence": "NLYIQWLKDGGPSSGRPPPS", "entity_type": "protein"}])  # Trp-cage mini-protein

batch_inputs = BioEmuInput(complexes=[chignolin, trp_cage])
batch_config = BioEmuConfig(num_samples=20, verbose=False)
batch_result = run_bioemu(batch_inputs, batch_config)

# Quantify conformational diversity via the radius-of-gyration spread across the ensemble.
for idx, ensemble in enumerate(batch_result.ensembles):
    metrics = run_structure_metrics(
        StructureMetricsInput(structures=ensemble.structures), StructureMetricsConfig()
    )
    rgs = [m.gyration_radius for m in metrics.metrics]
    print(
        f"Complex {idx}: {len(ensemble.structures)} conformations, "
        f"Rg {min(rgs):.1f}-{max(rgs):.1f} A (spread {max(rgs) - min(rgs):.2f} A)"
    )

## Export Results

Ensemble outputs can be saved in two formats. PDB format produces a directory tree with one PDB file per conformation, organized by ensemble, which is useful for visualization or as input to downstream tools. JSON format produces a single file containing all conformations, which is convenient for programmatic analysis.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

batch_result.export(name="bioemu_batch", export_path=output_dir, file_format="json")
print("Batch ensembles exported to example_output/bioemu_batch.json")